# Continuous-Time Markov Chain Exploration

This notebook runs the CTMC progression for the capstone project:

1. Global CTMC baseline
2. Clustered CTMC segmentation
3. Personalized neural-style CTMC rates
4. Benchmark model comparison

The code uses checkpointed parquet files from the local `data/` folder.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "requirements.txt").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src" / "models"))
sys.path.insert(0, str(PROJECT_ROOT / "src" / "visualizations"))
sys.path.insert(0, str(PROJECT_ROOT / ".codex_deps"))

import matplotlib.pyplot as plt
import pandas as pd

from ctmc import (
    CTMCData,
    ClusteredCTMC,
    GlobalCTMC,
    ModelComparison,
    NeuralRateCTMC,
    sample_pipeline,
)
from ctmc_submission import create_ctmc_submissions
from ctmc_plots import (
    plot_absorption_by_state,
    plot_calibration,
    plot_generator_heatmap,
    plot_metric_comparison,
    plot_top_transition_graph,
)
from tabular_submission import create_tabular_submissions

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# Increase this once the workflow is settled.
MAX_JOURNEYS = 50_000
N_CLUSTERS = 4

## Load Transition Data

Each row below is one observed transition: current action, next action, and elapsed time.

In [ ]:
data = CTMCData()
transitions = data.transition_table(max_journeys=MAX_JOURNEYS)

print(transitions.shape)
transitions.head()

## 1. Baseline Global CTMC

Estimate transition counts $N_{ij}$, time in state $T_i$, and global generator rates $q_{ij}=N_{ij}/T_i$.

In [ ]:
global_ctmc = GlobalCTMC().fit(transitions)

print("Q shape:", global_ctmc.Q_.shape)
display(global_ctmc.transition_counts_.head())
display(global_ctmc.time_in_state_.sort_values(ascending=False).head(10).rename("seconds_in_state"))
display(global_ctmc.top_rates(20))

plot_generator_heatmap(global_ctmc, RESULTS_DIR / "ctmc_generator_heatmap.png")
plot_top_transition_graph(global_ctmc, RESULTS_DIR / "ctmc_top_transition_graph.png", n_edges=25)
absorption_by_state = plot_absorption_by_state(global_ctmc, RESULTS_DIR / "ctmc_absorption_by_state.png")
absorption_by_state.to_csv(RESULTS_DIR / "ctmc_absorption_by_state.csv", index=False)
display(absorption_by_state)

In [ ]:
q_abs = global_ctmc.Q_.copy()
for state in q_abs.index:
    q_abs.loc[state, state] = 0

plt.figure(figsize=(10, 8))
plt.imshow(q_abs, aspect="auto")
plt.colorbar(label="rate")
plt.title("Global CTMC off-diagonal transition rates")
plt.xlabel("to state index")
plt.ylabel("from state index")
plt.tight_layout()
plt.show()

In [ ]:
# P(t) = exp(Qt): probability of being in each state after a horizon.
one_day = 24 * 60 * 60
p_1day = global_ctmc.transition_probability(one_day)
p_1day.head()

## 2. Segmentation: Clustered CTMC

Cluster users using action counts/proportions, observed time in states, time to key actions, and prefix metadata. The clusterer does not use the purchase label.

In [ ]:
clustered = ClusteredCTMC(n_clusters=N_CLUSTERS, random_state=42).fit(transitions)
cluster_summary = clustered.cluster_summary()
display(cluster_summary)

cluster_summary.plot.bar(x="cluster", y="n_journeys", legend=False, figsize=(6, 4))
plt.title("Journey clusters")
plt.ylabel("n journeys")
plt.tight_layout()
plt.show()

In [ ]:
for cluster_id, model in clustered.models_.items():
    print(f"Cluster {cluster_id}")
    display(model.top_rates(10))

## 3. Neural Exponential-Rate CTMC

The neural rate model learns a customer-dependent success intensity, without predicting the next state:

$$P(T_{success} \le t \mid x) = 1 - \exp(-\lambda(x)t).$$

This is simpler than a full neural CTMC, but it directly estimates the exponential waiting-time rate relevant to the 60-day success horizon.

In [ ]:
features = clustered.feature_builder.transform(transitions)
display(features.head())

neural_training = data.load_neural_rate_training_features(max_rows=100_000)
neural_ctmc = NeuralRateCTMC(hidden_layer_sizes=(64, 32), random_state=42)
neural_ctmc.fit(neural_training)

example_rows = neural_training.head(10)
pd.DataFrame({
    "id": example_rows["id"],
    "label": example_rows["label"],
    "lambda_hat": neural_ctmc.predict_lambda(example_rows),
    "p_success_60d": neural_ctmc.predict_success_probability(example_rows),
})

## 4. Model Comparison

Compare against standard predictive baselines on the truncated training features created by the pipeline.

In [ ]:
training_df = data.load_training_features(max_rows=100_000)
print(training_df.shape)
training_df.head()

In [ ]:
comparison = ModelComparison(random_state=42).run(training_df)
comparison.to_csv(RESULTS_DIR / "tabular_baseline_comparison.csv", index=False)
display(comparison)

comparison.plot.bar(x="model", y=["roc_auc", "average_precision"], figsize=(8, 4))
plt.ylim(0, 1)
plt.title("Benchmark comparison")
plt.tight_layout()
plt.show()

comparison.plot.bar(x="model", y=["log_loss", "brier_score"], figsize=(8, 4))
plt.title("Benchmark probability loss")
plt.tight_layout()
plt.show()

## 5. CTMC vs Baselines

This cell evaluates the three CTMC variants on journeys in the sampled transition table whose labels are available in the engineered training data. It is an exploration metric, not the official Kaggle score.

**Leakage warning:** if CTMC features are built from full completed journeys, successful journeys can expose final state `28` (`order_shipped`). Treat CTMC scores here as diagnostic unless the evaluation uses the same truncated/open-journey observation window as Kaggle.

In [ ]:
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score

label_lookup = data.load_binary_labels()
ctmc_eval = features.merge(label_lookup, on="id", how="inner")
ctmc_eval["state"] = ctmc_eval["current_state"]

ctmc_rows = []
ctmc_predictions = {
    "ctmc_global": global_ctmc.absorption_probability(ctmc_eval["current_state"]),
    "ctmc_clustered": clustered.predict_success_probability(ctmc_eval, fallback_model=global_ctmc),
    "ctmc_neural_rate": neural_ctmc.predict_success_probability(ctmc_eval),
}

for name, probs in ctmc_predictions.items():
    ctmc_rows.append({
        "model": name,
        "roc_auc": roc_auc_score(ctmc_eval["label"], probs),
        "average_precision": average_precision_score(ctmc_eval["label"], probs),
        "log_loss": log_loss(ctmc_eval["label"], probs, labels=[0, 1]),
        "brier_score": brier_score_loss(ctmc_eval["label"], probs),
    })

ctmc_comparison = pd.concat([comparison, pd.DataFrame(ctmc_rows)], ignore_index=True)
ctmc_comparison = ctmc_comparison.sort_values("roc_auc", ascending=False)
ctmc_comparison.to_csv(RESULTS_DIR / "ctmc_vs_baselines_comparison.csv", index=False)
plot_metric_comparison(ctmc_comparison, RESULTS_DIR / "ctmc_vs_baselines_metrics.png")
display(ctmc_comparison)

ctmc_comparison.plot.bar(x="model", y=["roc_auc", "average_precision"], figsize=(10, 4))
plt.ylim(0, 1)
plt.title("CTMC variants vs tabular baselines")
plt.tight_layout()
plt.show()

ctmc_comparison.sort_values("log_loss").plot.bar(x="model", y=["log_loss", "brier_score"], figsize=(10, 4))
plt.title("CTMC variants vs baselines: probability loss")
plt.tight_layout()
plt.show()

for name, probs in ctmc_predictions.items():
    calibration = plot_calibration(
        ctmc_eval["label"],
        probs,
        RESULTS_DIR / f"{name}_calibration.png",
        title=f"{name} calibration",
    )
    calibration.to_csv(RESULTS_DIR / f"{name}_calibration.csv", index=False)

## 6. Advanced CTMC Methods

Three additions motivated by the literature review:

| Method | Key idea |
|---|---|
| **LaplaceGlobalCTMC** | Add α pseudo-counts to every N_ij before computing rates — tames noisy rate estimates for rare state pairs |
| **WeibullSemiMarkovCTMC** | Relax the exponential holding-time assumption; fit per-state Weibull distributions and compute absorption via Monte Carlo |
| **CalibratedCTMC** | Post-hoc isotonic regression maps raw predicted probabilities to empirical success rates — directly minimises brier score |

All three are in `src/models/ctmc_advanced.py`.

In [ ]:
from ctmc_advanced import (
    LaplaceGlobalCTMC,
    WeibullSemiMarkovCTMC,
    CalibratedCTMC,
    compare_advanced_models,
)

# Reload labels aligned to the features we already built.
label_lookup = data.load_binary_labels()
eval_df = features.merge(label_lookup, on="id", how="inner").reset_index(drop=True)
eval_labels = eval_df["label"]

print(f"Eval set: {len(eval_df)} journeys  |  base rate: {eval_labels.mean():.3f}")

### 6a. LaplaceGlobalCTMC — Smoothed Generator Matrix

Adds α = 0.5 (Jeffreys prior) pseudo-counts to each N_ij.  Compare the top transition rates and absorption probabilities against the unsmoothed baseline.

In [ ]:
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

laplace_ctmc = LaplaceGlobalCTMC(alpha=0.5).fit(transitions)

raw_global   = global_ctmc.absorption_probability(eval_df["current_state"])
raw_laplace  = laplace_ctmc.absorption_probability(eval_df["current_state"])

y = eval_labels.to_numpy()

print("GlobalCTMC (unsmoothed)")
print(f"  brier={brier_score_loss(y, raw_global):.4f}  "
      f"logloss={log_loss(y, raw_global, labels=[0,1]):.4f}  "
      f"auc={roc_auc_score(y, raw_global):.4f}")

print("\nLaplaceGlobalCTMC (α=0.5)")
print(f"  brier={brier_score_loss(y, raw_laplace):.4f}  "
      f"logloss={log_loss(y, raw_laplace, labels=[0,1]):.4f}  "
      f"auc={roc_auc_score(y, raw_laplace):.4f}")

# Show how smoothing affects the predicted probability distribution.
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 3))
plt.hist(raw_global,  bins=50, alpha=0.6, label="Global (unsmoothed)")
plt.hist(raw_laplace, bins=50, alpha=0.6, label="Laplace (α=0.5)")
plt.xlabel("Predicted P(success)")
plt.ylabel("Count")
plt.title("Effect of Laplace smoothing on predicted probability distribution")
plt.legend()
plt.tight_layout()
plt.show()

### 6b. WeibullSemiMarkovCTMC — Non-Exponential Holding Times

Standard CTMC forces exponential dwell times (constant hazard rate per state).  Customer actions cluster in time — dwell times are closer to Weibull or lognormal.  This model fits a Weibull distribution per state and samples paths via Monte Carlo.

`n_sims=300` gives a good bias-variance trade-off; increase for a smoother estimate.

In [ ]:
# Fit Weibull parameters per state and inspect a few.
weibull_ctmc = WeibullSemiMarkovCTMC(n_sims=300, random_state=42).fit(transitions)

weibull_summary = pd.DataFrame(
    [(state, shape, scale) for state, (shape, scale) in weibull_ctmc.weibull_params_.items()],
    columns=["state", "weibull_shape", "weibull_scale_seconds"],
).sort_values("weibull_shape")

print("States with shape < 1 (decreasing hazard — burst behavior):")
display(weibull_summary[weibull_summary["weibull_shape"] < 1].head(8))
print("\nStates with shape > 1 (increasing hazard — aging behavior):")
display(weibull_summary[weibull_summary["weibull_shape"] > 1].tail(8))

weibull_summary["weibull_shape"].hist(bins=20, figsize=(6, 3))
plt.xlabel("Weibull shape (κ)")
plt.title("Per-state Weibull shape — κ=1 is exponential, κ<1 is bursty, κ>1 is aging")
plt.tight_layout()
plt.show()

In [ ]:
# Absorption probabilities from Weibull semi-Markov (takes ~1 min for 300 sims).
print("Running Monte Carlo absorption (this may take a minute)...")
raw_weibull = weibull_ctmc.absorption_probability(eval_df["current_state"])

print("WeibullSemiMarkovCTMC")
print(f"  brier={brier_score_loss(y, raw_weibull):.4f}  "
      f"logloss={log_loss(y, raw_weibull, labels=[0,1]):.4f}  "
      f"auc={roc_auc_score(y, raw_weibull):.4f}")

plt.figure(figsize=(8, 3))
plt.hist(raw_global,  bins=50, alpha=0.5, label="GlobalCTMC (exponential)")
plt.hist(raw_weibull, bins=50, alpha=0.5, label="WeibullSemiMarkov")
plt.xlabel("Predicted P(success)")
plt.title("Exponential vs Weibull holding-time model: predicted probability distribution")
plt.legend()
plt.tight_layout()
plt.show()

### 6c. CalibratedCTMC — Isotonic Calibration

Isotonic regression finds the best-fitting monotone mapping from raw predicted probabilities to true success rates, minimising squared error (= brier score) on a calibration split.  Applied after any CTMC predictor.

In [ ]:
from sklearn.model_selection import train_test_split

# Split eval set: 70% calibration fit, 30% final evaluation.
idx_cal, idx_test = train_test_split(
    np.arange(len(eval_df)), test_size=0.3, random_state=42, stratify=eval_labels
)
feat_cal_  = eval_df.iloc[idx_cal].reset_index(drop=True)
feat_test_ = eval_df.iloc[idx_test].reset_index(drop=True)
lab_cal_   = eval_labels.iloc[idx_cal].reset_index(drop=True)
lab_test_  = eval_labels.iloc[idx_test].to_numpy()

# Calibrate both GlobalCTMC and LaplaceGlobalCTMC.
cal_global_  = CalibratedCTMC(global_ctmc).fit_calibration(feat_cal_, lab_cal_)
cal_laplace_ = CalibratedCTMC(laplace_ctmc).fit_calibration(feat_cal_, lab_cal_)

probs_raw_test     = global_ctmc.absorption_probability(feat_test_["current_state"])
probs_cal_global   = cal_global_.predict(feat_test_)
probs_cal_laplace  = cal_laplace_.predict(feat_test_)

for name, probs in [
    ("GlobalCTMC (raw)",          probs_raw_test),
    ("GlobalCTMC (calibrated)",   probs_cal_global),
    ("LaplaceGlobalCTMC (calibrated)", probs_cal_laplace),
]:
    print(f"{name:40s}  brier={brier_score_loss(lab_test_, probs):.4f}  "
          f"auc={roc_auc_score(lab_test_, probs):.4f}")

# Calibration curves.
from sklearn.calibration import calibration_curve
fig, ax = plt.subplots(figsize=(6, 5))
for name, probs, color in [
    ("GlobalCTMC (raw)",          probs_raw_test,    "tab:blue"),
    ("GlobalCTMC (calibrated)",   probs_cal_global,  "tab:orange"),
    ("Laplace (calibrated)",      probs_cal_laplace, "tab:green"),
]:
    frac_pos, mean_pred = calibration_curve(lab_test_, probs, n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, marker="o", label=name, color=color)
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction positive")
ax.set_title("Calibration curves: raw vs isotonic-calibrated CTMC")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 6d. Fixed NeuralRateCTMC — Censored Exponential Likelihood

The original NeuralRateCTMC assigned label=0 rows a fictitious tiny hazard (1 / 10×horizon), which is theoretically wrong.  The fixed version trains on the proper censored exponential log-likelihood:

- **Event** (label=1): log λ − λ t_event
- **Censored** (label=0): −λ t_horizon

This treats did-not-ship journeys as right-censored, not as observations at a fake time.

In [ ]:
neural_training = data.load_neural_rate_training_features(max_rows=100_000)

neural_fixed = NeuralRateCTMC(hidden_layer_sizes=(64, 32), random_state=42)
neural_fixed.fit(neural_training)

# Align eval features with neural feature columns.
neural_eval = neural_training.merge(label_lookup, on="id", how="inner")
y_neural = neural_eval["label"].to_numpy()
probs_neural = neural_fixed.predict_success_probability(neural_eval)

print("Fixed NeuralRateCTMC (censored exponential likelihood)")
print(f"  brier={brier_score_loss(y_neural, probs_neural):.4f}  "
      f"logloss={log_loss(y_neural, probs_neural, labels=[0,1]):.4f}  "
      f"auc={roc_auc_score(y_neural, probs_neural):.4f}")

plt.figure(figsize=(8, 3))
plt.hist(probs_neural, bins=50)
plt.xlabel("Predicted P(success within 60 days)")
plt.title("Fixed NeuralRateCTMC — predicted probability distribution")
plt.tight_layout()
plt.show()

### 6e. Full Method Comparison

All methods on the same held-out eval set, sorted by brier score.

In [ ]:
from sklearn.metrics import average_precision_score

def metrics(name, probs, y_true):
    p = np.clip(probs, 1e-6, 1 - 1e-6)
    return {
        "model": name,
        "brier_score": brier_score_loss(y_true, p),
        "log_loss": log_loss(y_true, p, labels=[0, 1]),
        "roc_auc": roc_auc_score(y_true, p),
        "avg_precision": average_precision_score(y_true, p),
    }

rows = [
    metrics("GlobalCTMC",                raw_global,       y),
    metrics("LaplaceGlobalCTMC",         raw_laplace,      y),
    metrics("GlobalCTMC (calibrated)",   cal_global_.predict(eval_df),  y),
    metrics("LaplaceGlobalCTMC (cal.)",  cal_laplace_.predict(eval_df), y),
    metrics("WeibullSemiMarkov",         raw_weibull,      y),
    metrics("ClusteredCTMC",
            clustered.predict_success_probability(eval_df, fallback_model=global_ctmc), y),
    metrics("NeuralRateCTMC (fixed)",    probs_neural,     y_neural),
]

summary = pd.DataFrame(rows).sort_values("brier_score").reset_index(drop=True)
summary.to_csv(RESULTS_DIR / "advanced_ctmc_comparison.csv", index=False)
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
summary.plot.barh(x="model", y="brier_score", ax=axes[0], legend=False)
axes[0].set_title("Brier score (lower is better)")
axes[0].invert_yaxis()
summary.plot.barh(x="model", y="roc_auc", ax=axes[1], legend=False)
axes[1].set_title("ROC AUC (higher is better)")
axes[1].invert_yaxis()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "advanced_ctmc_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Write Kaggle Submission CSVs

This writes the model outputs to `results/`. The cell expects `data/open_journeys1.csv` to exist. If `data/open_journeys1_flattened_all0.csv` is missing, the submission helper creates the all-zero ID template from the open journeys file.

In [ ]:
TEST_EVENTS = PROJECT_ROOT / "data" / "open_journeys1.csv"
SAMPLE_TEMPLATE = PROJECT_ROOT / "data" / "open_journeys1_flattened_all0.csv"

if not TEST_EVENTS.exists():
    print(f"Missing {TEST_EVENTS}. Add the Kaggle open journey event file before writing submission CSVs.")
else:
    ctmc_outputs = create_ctmc_submissions(
        test_events_path=TEST_EVENTS,
        sample_path=SAMPLE_TEMPLATE,
        output_dir=RESULTS_DIR,
        max_train_journeys=MAX_JOURNEYS,
        n_clusters=N_CLUSTERS,
        neural_transition_limit=75_000,
    )
    tabular_outputs = create_tabular_submissions(
        test_events_path=TEST_EVENTS,
        sample_path=SAMPLE_TEMPLATE,
        output_dir=RESULTS_DIR,
        max_train_rows=300_000,
    )
    print("Wrote submission files:")
    for path in sorted(RESULTS_DIR.glob("*_submission.csv")):
        print(path.name)


## Decision Notes

- Global CTMC: most interpretable and gives direct time-to-event dynamics.
- Clustered CTMC: keeps interpretability while capturing segment-level behavior.
- Personalized rates: more flexible, but less transparent and more tuning-sensitive.
- Tree/boosting baselines: likely stronger for raw prediction on engineered tabular features.
- Transformer/sequence model: likely best if the team has time to build a proper sequence validation setup.